# Finance Guidance Assistant — RAG Prototype (Colab)

Run all cells top to bottom. Make sure a GPU runtime is selected:
**Runtime → Change runtime type → Hardware accelerator → GPU (T4)**

This notebook:
1. Installs dependencies
2. Gets the project code (clone from GitHub, or upload manually)
3. Builds the FAISS index from the finance knowledge base
4. Runs a quick retrieval sanity check
5. Launches the Gradio app (shareable public link)
6. Runs the evaluation suite and shows results


# 1. Dependencies

In [1]:
!pip install sentence-transformers faiss-cpu transformers torch gradio accelerate --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 97.5 MB/s eta 0:00:00


# 2. Clone from GitHub

In [2]:
GITHUB_REPO_URL = "https://github.com/SheerinMM/finance-guidance-rag.git"

!git clone {GITHUB_REPO_URL}
%cd finance-rag

Cloning into 'finance-guidance-rag'...
remote: Enumerating objects: 55, done.
remote: Counting objects: 100% (55/55), done.
remote: Compressing objects: 100% (35/35), done.
remote: Total 55 (delta 19), reused 50 (delta 17), pack-reused 0 (from 0)
Receiving objects: 100% (55/55), 38.35 KiB | 4.79 MiB/s, done.
Resolving deltas: 100% (19/19), done.
[Errno 2] No such file or directory: 'finance-rag'
/content


## 3. Build the FAISS index

In [3]:
!pwd
!ls

/content
finance-guidance-rag  sample_data


In [4]:
%cd finance-guidance-rag

/content/finance-guidance-rag


In [5]:
!ls -la
!ls -la src

total 44
drwxr-xr-x 7 root root 4096 Jul  4 11:12 .
drwxr-xr-x 1 root root 4096 Jul  4 11:12 ..
drwxr-xr-x 2 root root 4096 Jul  4 11:12 data
drwxr-xr-x 2 root root 4096 Jul  4 11:12 eval
drwxr-xr-x 8 root root 4096 Jul  4 11:12 .git
-rw-r--r-- 1 root root  224 Jul  4 11:12 .gitignore
drwxr-xr-x 2 root root 4096 Jul  4 11:12 notebooks
-rw-r--r-- 1 root root 5985 Jul  4 11:12 README.md
-rw-r--r-- 1 root root  114 Jul  4 11:12 requirements.txt
drwxr-xr-x 2 root root 4096 Jul  4 11:12 src
total 32
drwxr-xr-x 2 root root 4096 Jul  4 11:12 .
drwxr-xr-x 7 root root 4096 Jul  4 11:12 ..
-rw-r--r-- 1 root root 2422 Jul  4 11:12 app.py
-rw-r--r-- 1 root root 2275 Jul  4 11:12 chunking.py
-rw-r--r-- 1 root root 2166 Jul  4 11:12 embed_index.py
-rw-r--r-- 1 root root 2784 Jul  4 11:12 generate.py
-rw-r--r-- 1 root root 1475 Jul  4 11:12 retrieve.py
-rw-r--r-- 1 root root 3667 Jul  4 11:12 safeguard.py


In [6]:
import os, sys

os.chdir('/content/finance-guidance-rag')  # force absolute path, no ambiguity
sys.path.insert(0, os.path.join(os.getcwd(), 'src'))  # use absolute path too

print("cwd:", os.getcwd())
print("src exists:", os.path.exists('src/chunking.py'))
print("sys.path[0:2]:", sys.path[0:2])

import chunking
print("Import worked:", chunking.__file__)

cwd: /content/finance-guidance-rag
src exists: True
sys.path[0:2]: ['/content/finance-guidance-rag/src', '/content']
Import worked: /content/finance-guidance-rag/src/chunking.py


In [7]:
import os, sys

os.chdir('/content/finance-guidance-rag')
sys.path.insert(0, os.path.join(os.getcwd(), 'src'))

from chunking import build_chunk_corpus
from embed_index import build_index, save_index

corpus = build_chunk_corpus('data/finance_kb.json')
print(f"Loaded {len(corpus)} chunks from knowledge base.")

index, model, corpus = build_index(corpus)
save_index(index, corpus, 'data/faiss_index.bin', 'data/corpus.json')
print(f"Built FAISS index with {index.ntotal} vectors (dim={index.d}).")

Loaded 27 chunks from knowledge base.


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Built FAISS index with 27 vectors (dim=384).


## 4. Quick retrieval sanity check

In [8]:
from retrieve import retrieve

test_queries = [
    "When can I take money out of my pension?",
    "What's the difference between saving and investing?",
    "How do I improve my credit score?",
]

for q in test_queries:
    print(f"\nQuery: {q}")
    results = retrieve(q, index, model, corpus, top_k=3)
    for r in results:
        print(f"  [{r['score']:.3f}] {r['title']} ({r['category']})")


Query: When can I take money out of my pension?
  [0.532] When can I normally access my pension? (Pensions)
  [0.531] What are my options when taking a defined contribution pension? (Pensions)
  [0.469] Should I combine multiple pension pots? (Pensions)

Query: What's the difference between saving and investing?
  [0.806] What is the difference between saving and investing? (Investing)
  [0.464] How does inflation affect savings? (Economy)
  [0.455] What is an ISA? (Savings)

Query: How do I improve my credit score?
  [0.685] How can I improve my credit score? (Credit)
  [0.490] What is a credit score and how is it used? (Credit)
  [0.255] What should I do if I am struggling with debt repayments? (Debt)


## 5. Safeguard

In [9]:
from safeguard import check_safeguards

test_cases = [
    "Should I transfer my pension into a SIPP given my situation?",
    "What is a Lifetime ISA?",
]

for q in test_cases:
    retrieved = retrieve(q, index, model, corpus, top_k=3)
    blocked, message = check_safeguards(q, retrieved)
    print(f"Query: {q}")
    print(f"  Blocked: {blocked}")
    if blocked:
        print(f"  Message: {message}")
    print()

Query: Should I transfer my pension into a SIPP given my situation?
  Blocked: True
  Message: I can't give personalised financial advice — that requires an FCA-regulated adviser who can assess your individual circumstances. What I can do is explain general guidance on this topic. For personalised recommendations, consider a free Pension Wise appointment (if pension-related) or a regulated financial adviser. Would you like me to explain the general options instead?

Query: What is a Lifetime ISA?
  Blocked: False



## 6. Model and test end-to-end

In [10]:
%%writefile src/generate.py
"""
generate.py
Builds a grounded prompt from the retrieved chunks and calls a pretrained LLM
to produce the final answer, with citations back to source titles.
"""

from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

GENERATION_MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

SYSTEM_PROMPT = (
    "You are a financial guidance assistant for UK consumers. Answer ONLY using "
    "the information provided in the context below. If the context does not "
    "contain enough information to answer, say so clearly rather than guessing. "
    "Do not give personalised financial advice or recommend specific products for "
    "the user's individual situation — explain general options instead. "
    "Keep answers concise and in plain English."
)


def build_prompt(query, retrieved_chunks):
    context = "\n\n".join(
        f"[Source: {c['title']}]\n{c['text']}" for c in retrieved_chunks
    )
    return (
        f"{SYSTEM_PROMPT}\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {query}\n\n"
        f"Answer:"
    )


class Generator:
    def __init__(self, model_name=GENERATION_MODEL_NAME):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
            device_map="auto" if torch.cuda.is_available() else None,
        )

    def generate(self, query, retrieved_chunks, max_new_tokens=200):
        prompt = build_prompt(query, retrieved_chunks)
        messages = [{"role": "user", "content": prompt}]

        prompt_text = self.tokenizer.apply_chat_template(
            messages, add_generation_prompt=True, tokenize=False
        )
        encoded = self.tokenizer(prompt_text, return_tensors="pt")
        input_ids = encoded["input_ids"]
        attention_mask = encoded["attention_mask"]

        if torch.cuda.is_available():
            input_ids = input_ids.to(self.model.device)
            attention_mask = attention_mask.to(self.model.device)

        outputs = self.model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=max_new_tokens,
            temperature=0.3,
            do_sample=True,
            pad_token_id=self.tokenizer.eos_token_id,
        )
        response = self.tokenizer.decode(
            outputs[0][input_ids.shape[-1]:], skip_special_tokens=True
        )
        return response.strip()


def format_answer_with_citations(answer, retrieved_chunks):
    sources = sorted(set(c["title"] for c in retrieved_chunks))
    citation_block = "\n\nSources:\n" + "\n".join(f"- {s}" for s in sources)
    return answer + citation_block

Overwriting src/generate.py


In [11]:
from generate import Generator, format_answer_with_citations

generator = Generator()  # downloads Qwen2.5-1.5B-Instruct on first run

query = "What is a Lifetime ISA?"
retrieved = retrieve(query, index, model, corpus, top_k=3)
blocked, message = check_safeguards(query, retrieved)

if blocked:
    print(message)
else:
    answer = generator.generate(query, retrieved)
    print(format_answer_with_citations(answer, retrieved))

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

A Lifetime ISA (LISA) is a savings account designed specifically for individuals aged between 18 and 39. It allows savers to make contributions up to a certain annual limit, with the government adding a 25% bonus on top of these contributions. Unlike regular savings accounts, withdrawals from a LISA for non-home-related purposes typically result in a government withdrawal charge, potentially reducing the amount received compared to what was initially saved.

Sources:
- What is a Lifetime ISA?
- What is a defined benefit pension?
- What is an ISA?


## 7. Gradio app

In [12]:
import gradio as gr

def answer_question(query, history=None):
    if not query or not query.strip():
        return "Please enter a question."
    retrieved = retrieve(query, index, model, corpus, top_k=3)
    blocked, message = check_safeguards(query, retrieved)
    if blocked:
        return message
    raw_answer = generator.generate(query, retrieved)
    return format_answer_with_citations(raw_answer, retrieved)

demo = gr.ChatInterface(
    fn=answer_question,
    title="Finance Guidance Assistant",
    description="Ask general questions about UK pensions, savings, debt, mortgages, credit, and investing. This is guidance, not regulated financial advice.",
    examples=[
        "What is the difference between a defined contribution and defined benefit pension?",
        "What is a Lifetime ISA?",
        "How can I recognise a pension scam?",
        "Should I move my pension into a SIPP given my situation?",
    ],
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f8584072864cec2f85.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## 8. Evaluation suite



In [13]:
%cd ..
!python eval/evaluate.py
%cd finance-rag

/content
python3: can't open file '/content/eval/evaluate.py': [Errno 2] No such file or directory
[Errno 2] No such file or directory: 'finance-rag'
/content


## 9. Evaluation results




In [14]:
import os
os.chdir('/content/finance-guidance-rag')
!python eval/evaluate.py

Traceback (most recent call last):
  File "/content/finance-guidance-rag/eval/evaluate.py", line 29, in <module>
    from embed_index import load_index
ModuleNotFoundError: No module named 'embed_index'


In [15]:
%%writefile eval/evaluate.py
"""
evaluate.py
Runs the 20-question test set through the RAG pipeline and produces the
quantitative evidence needed for Part B's evaluation section.
"""

import json
import time
import csv
import sys
import os

_THIS_DIR = os.path.dirname(os.path.abspath(__file__))
_REPO_ROOT = os.path.dirname(_THIS_DIR)
_SRC_DIR = os.path.join(_REPO_ROOT, "src")
if _SRC_DIR not in sys.path:
    sys.path.insert(0, _SRC_DIR)

from embed_index import load_index
from retrieve import retrieve
from safeguard import check_safeguards
from generate import Generator, format_answer_with_citations

TOP_K = 3

SAFEGUARD_TEST_QUERIES = [
    "Should I transfer my whole pension into a SIPP?",
    "What should I invest in given my situation?",
    "Which mortgage is best for me?",
]


def run_retrieval_evaluation(test_set, index, embed_model, corpus):
    hits = 0
    latencies = []
    rows = []
    for item in test_set:
        start = time.perf_counter()
        results = retrieve(item["question"], index, embed_model, corpus, top_k=TOP_K)
        elapsed = time.perf_counter() - start
        latencies.append(elapsed)
        retrieved_ids = [r["doc_id"] for r in results]
        hit = item["expected_doc_id"] in retrieved_ids
        hits += int(hit)
        rows.append({
            "question": item["question"],
            "expected_doc_id": item["expected_doc_id"],
            "retrieved_doc_ids": ";".join(retrieved_ids),
            "retrieval_hit": hit,
            "top_score": round(results[0]["score"], 3) if results else None,
            "retrieval_latency_sec": round(elapsed, 3),
        })
    accuracy = hits / len(test_set)
    avg_latency = sum(latencies) / len(latencies)
    return accuracy, avg_latency, rows


def run_generation_evaluation(test_set, index, embed_model, corpus, generator):
    rows = []
    for item in test_set:
        start = time.perf_counter()
        retrieved = retrieve(item["question"], index, embed_model, corpus, top_k=TOP_K)
        blocked, message = check_safeguards(item["question"], retrieved)
        if blocked:
            answer = message
        else:
            raw = generator.generate(item["question"], retrieved)
            answer = format_answer_with_citations(raw, retrieved)
        elapsed = time.perf_counter() - start
        rows.append({
            "question": item["question"],
            "ground_truth": item["ground_truth"],
            "generated_answer": answer,
            "safeguard_triggered": blocked,
            "total_latency_sec": round(elapsed, 3),
            "manual_grade": "",
        })
    return rows


def run_safeguard_evaluation(queries, index, embed_model, corpus):
    rows = []
    for q in queries:
        retrieved = retrieve(q, index, embed_model, corpus, top_k=TOP_K)
        blocked, message = check_safeguards(q, retrieved)
        rows.append({"question": q, "blocked": blocked, "message": message})
    return rows


def write_csv(rows, path):
    if not rows:
        return
    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=rows[0].keys())
        writer.writeheader()
        writer.writerows(rows)


if __name__ == "__main__":
    test_questions_path = os.path.join(_THIS_DIR, "test_questions.json")
    index_path = os.path.join(_REPO_ROOT, "data", "faiss_index.bin")
    corpus_path = os.path.join(_REPO_ROOT, "data", "corpus.json")

    with open(test_questions_path) as f:
        test_set = json.load(f)

    print("Loading index and models...")
    index, embed_model, corpus = load_index(index_path, corpus_path)
    generator = Generator()

    print("\n--- Retrieval evaluation ---")
    accuracy, avg_latency, retrieval_rows = run_retrieval_evaluation(test_set, index, embed_model, corpus)
    print(f"Retrieval accuracy @ top-{TOP_K}: {accuracy:.1%}")
    print(f"Average retrieval latency: {avg_latency*1000:.1f} ms")
    write_csv(retrieval_rows, os.path.join(_THIS_DIR, "retrieval_results.csv"))

    print("\n--- Full pipeline (generation) evaluation ---")
    generation_rows = run_generation_evaluation(test_set, index, embed_model, corpus, generator)
    write_csv(generation_rows, os.path.join(_THIS_DIR, "generation_results.csv"))
    print("Saved to eval/generation_results.csv")

    print("\n--- Safeguard evaluation ---")
    safeguard_rows = run_safeguard_evaluation(SAFEGUARD_TEST_QUERIES, index, embed_model, corpus)
    for r in safeguard_rows:
        print(f"  Blocked={r['blocked']}: {r['question']}")
    write_csv(safeguard_rows, os.path.join(_THIS_DIR, "safeguard_results.csv"))

    print("\nDone.")

Overwriting eval/evaluate.py


In [16]:
!python eval/evaluate.py

Loading index and models...
Loading weights: 100% 103/103 [00:00<00:00, 879.38it/s]
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 338/338 [00:12<00:00, 27.16it/s]

--- Retrieval evaluation ---
Retrieval accuracy @ top-3: 100.0%
Average retrieval latency: 73.4 ms

--- Full pipeline (generation) evaluation ---
Saved to eval/generation_results.csv

--- Safeguard evaluation ---
  Blocked=True: Should I transfer my whole pension into a SIPP?
  Blocked=True: What should I invest in given my situation?
  Blocked=False: Which mortgage is best for me?

Done.


In [17]:
%%writefile src/safeguard.py
"""
safeguard.py
Implements the required safeguard for this prototype: a guardrail that detects
requests for personalised, regulated financial advice and refuses to answer
directly, returning a standard disclaimer instead of a generated response.
"""

import re

ADVICE_PATTERNS = [
    r"\bshould i\b",
    r"\bwhat should i do\b",
    r"\bis it (a )?good idea for me\b",
    r"\brecommend\b.*\b(for me|to me)\b",
    r"\bwhich\b.*\b(should i|is best for me|is right for me|suits me)\b",
    r"\bgiven my (situation|circumstances|salary|age|pension)\b",
    r"\bwhat would you do\b",
]

DISCLAIMER = (
    "I can't give personalised financial advice — that requires an FCA-regulated "
    "adviser who can assess your individual circumstances. What I can do is explain "
    "general guidance on this topic. For personalised recommendations, consider a "
    "free Pension Wise appointment (if pension-related) or a regulated financial "
    "adviser. Would you like me to explain the general options instead?"
)

LOW_CONFIDENCE_MESSAGE = (
    "I don't have enough relevant information in my knowledge base to answer that "
    "confidently. Rather than guess, I'd rather say I don't know than risk giving "
    "you an inaccurate answer on a financial matter."
)

MIN_RETRIEVAL_CONFIDENCE = 0.35


def is_advice_request(query):
    q = query.lower()
    return any(re.search(pattern, q) for pattern in ADVICE_PATTERNS)


def check_safeguards(query, retrieved_chunks):
    if is_advice_request(query):
        return True, DISCLAIMER
    if not retrieved_chunks or retrieved_chunks[0]["score"] < MIN_RETRIEVAL_CONFIDENCE:
        return True, LOW_CONFIDENCE_MESSAGE
    return False, None

Overwriting src/safeguard.py


In [ ]:
import sys
if 'safeguard' in sys.modules:
    del sys.modules['safeguard']

!python eval/evaluate.py

Loading index and models...
Loading weights: 100% 103/103 [00:00<00:00, 1053.21it/s]
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 338/338 [00:10<00:00, 30.76it/s]

--- Retrieval evaluation ---
Retrieval accuracy @ top-3: 100.0%
Average retrieval latency: 29.0 ms

--- Full pipeline (generation) evaluation ---
Saved to eval/generation_results.csv

--- Safeguard evaluation ---
  Blocked=True: Should I transfer my whole pension into a SIPP?
  Blocked=True: What should I invest in given my situation?
  Blocked=True: Which mortgage is best for me?

Done.


In [ ]:
import pandas as pd

retrieval_df = pd.read_csv('eval/retrieval_results.csv')
print("Retrieval accuracy:", retrieval_df['retrieval_hit'].mean())
print("Average retrieval latency (s):", retrieval_df['retrieval_latency_sec'].mean())
retrieval_df

In [ ]:
generation_df = pd.read_csv('eval/generation_results.csv')
generation_df
# Fill in the 'manual_grade' column (correct / partial / incorrect) by reading each
# generated_answer against the ground_truth, then re-load this CSV to compute
# an overall accuracy percentage for your report.

In [ ]:
safeguard_df = pd.read_csv('eval/safeguard_results.csv')
safeguard_df

In [ ]:
import gradio as gr

def answer_question(query, history=None):
    if not query or not query.strip():
        return "Please enter a question."
    retrieved = retrieve(query, index, model, corpus, top_k=3)
    blocked, message = check_safeguards(query, retrieved)
    if blocked:
        return message
    raw_answer = generator.generate(query, retrieved)
    return format_answer_with_citations(raw_answer, retrieved)

demo = gr.ChatInterface(
    fn=answer_question,
    title="Finance Guidance Assistant",
    description="Ask general questions about UK pensions, savings, debt, mortgages, credit, and investing. This is guidance, not regulated financial advice.",
    examples=[
        "What is the difference between a defined contribution and defined benefit pension?",
        "What is a Lifetime ISA?",
        "How can I recognise a pension scam?",
        "Should I move my pension into a SIPP given my situation?",
    ],
)

demo.launch(share=True)

In [ ]:
import sys
if 'generate' in sys.modules:
    del sys.modules['generate']

# Patch just the one function rather than rewriting the whole file
import generate as _gen

def format_answer_with_citations(answer, retrieved_chunks, min_citation_score=0.30):
    relevant = [c for c in retrieved_chunks if c.get("score", 1.0) >= min_citation_score]
    if not relevant:
        relevant = retrieved_chunks[:1]
    sources = sorted(set(c["title"] for c in relevant))
    citation_block = "\n\nSources:\n" + "\n".join(f"- {s}" for s in sources)
    return answer + citation_block

_gen.format_answer_with_citations = format_answer_with_citations

In [ ]:
%%writefile src/generate.py
"""
generate.py
Builds a grounded prompt from the retrieved chunks and calls a pretrained LLM
to produce the final answer, with citations back to source titles.
"""

from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

GENERATION_MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

SYSTEM_PROMPT = (
    "You are a financial guidance assistant for UK consumers. Answer ONLY using "
    "the information provided in the context below. If the context does not "
    "contain enough information to answer, say so clearly rather than guessing. "
    "Do not give personalised financial advice or recommend specific products for "
    "the user's individual situation — explain general options instead. "
    "Keep answers concise and in plain English."
)


def build_prompt(query, retrieved_chunks):
    context = "\n\n".join(
        f"[Source: {c['title']}]\n{c['text']}" for c in retrieved_chunks
    )
    return (
        f"{SYSTEM_PROMPT}\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {query}\n\n"
        f"Answer:"
    )


class Generator:
    def __init__(self, model_name=GENERATION_MODEL_NAME):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
            device_map="auto" if torch.cuda.is_available() else None,
        )

    def generate(self, query, retrieved_chunks, max_new_tokens=200):
        prompt = build_prompt(query, retrieved_chunks)
        messages = [{"role": "user", "content": prompt}]
        prompt_text = self.tokenizer.apply_chat_template(
            messages, add_generation_prompt=True, tokenize=False
        )
        encoded = self.tokenizer(prompt_text, return_tensors="pt")
        input_ids = encoded["input_ids"]
        attention_mask = encoded["attention_mask"]
        if torch.cuda.is_available():
            input_ids = input_ids.to(self.model.device)
            attention_mask = attention_mask.to(self.model.device)
        outputs = self.model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=max_new_tokens,
            temperature=0.3,
            do_sample=True,
            pad_token_id=self.tokenizer.eos_token_id,
        )
        response = self.tokenizer.decode(
            outputs[0][input_ids.shape[-1]:], skip_special_tokens=True
        )
        return response.strip()


def format_answer_with_citations(answer, retrieved_chunks, min_citation_score=0.30):
    relevant = [c for c in retrieved_chunks if c.get("score", 1.0) >= min_citation_score]
    if not relevant:
        relevant = retrieved_chunks[:1]
    sources = sorted(set(c["title"] for c in relevant))
    citation_block = "\n\nSources:\n" + "\n".join(f"- {s}" for s in sources)
    return answer + citation_block

In [ ]:
import sys
for mod in ['generate']:
    if mod in sys.modules:
        del sys.modules[mod]

from generate import Generator, format_answer_with_citations
import gradio as gr

generator = Generator()  # reload — safe even though weights are cached locally now

def answer_question(query, history=None):
    if not query or not query.strip():
        return "Please enter a question."
    retrieved = retrieve(query, index, model, corpus, top_k=3)
    blocked, message = check_safeguards(query, retrieved)
    if blocked:
        return message
    raw_answer = generator.generate(query, retrieved)
    return format_answer_with_citations(raw_answer, retrieved)

demo = gr.ChatInterface(fn=answer_question, title="Finance Guidance Assistant")
demo.launch(share=True)

In [ ]:
scores = retrieve("What is a Lifetime ISA?", index, model, corpus, top_k=3)
for s in scores:
    print(f"{s['score']:.3f}  {s['title']}")

In [ ]:
import sys
if 'generate' in sys.modules:
    del sys.modules['generate']

from generate import format_answer_with_citations

# Direct test with the real scores from before
chunks = [
    {'title': 'What is a Lifetime ISA?', 'score': 0.694},
    {'title': 'What is an ISA?', 'score': 0.547},
    {'title': 'What is a defined benefit pension?', 'score': 0.338},
]
print(format_answer_with_citations("Test", chunks))

In [ ]:
%%writefile src/generate.py
"""
generate.py
Builds a grounded prompt from the retrieved chunks and calls a pretrained LLM
to produce the final answer, with citations back to source titles.
"""

from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

GENERATION_MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

SYSTEM_PROMPT = (
    "You are a financial guidance assistant for UK consumers. Answer ONLY using "
    "the information provided in the context below. If the context does not "
    "contain enough information to answer, say so clearly rather than guessing. "
    "Do not give personalised financial advice or recommend specific products for "
    "the user's individual situation — explain general options instead. "
    "Keep answers concise and in plain English."
)


def build_prompt(query, retrieved_chunks):
    context = "\n\n".join(
        f"[Source: {c['title']}]\n{c['text']}" for c in retrieved_chunks
    )
    return (
        f"{SYSTEM_PROMPT}\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {query}\n\n"
        f"Answer:"
    )


class Generator:
    def __init__(self, model_name=GENERATION_MODEL_NAME):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
            device_map="auto" if torch.cuda.is_available() else None,
        )

    def generate(self, query, retrieved_chunks, max_new_tokens=200):
        prompt = build_prompt(query, retrieved_chunks)
        messages = [{"role": "user", "content": prompt}]
        prompt_text = self.tokenizer.apply_chat_template(
            messages, add_generation_prompt=True, tokenize=False
        )
        encoded = self.tokenizer(prompt_text, return_tensors="pt")
        input_ids = encoded["input_ids"]
        attention_mask = encoded["attention_mask"]
        if torch.cuda.is_available():
            input_ids = input_ids.to(self.model.device)
            attention_mask = attention_mask.to(self.model.device)
        outputs = self.model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=max_new_tokens,
            temperature=0.3,
            do_sample=True,
            pad_token_id=self.tokenizer.eos_token_id,
        )
        response = self.tokenizer.decode(
            outputs[0][input_ids.shape[-1]:], skip_special_tokens=True
        )
        return response.strip()


def format_answer_with_citations(answer, retrieved_chunks, min_absolute_score=0.35, relative_margin=0.65):
    if not retrieved_chunks:
        return answer
    top_score = retrieved_chunks[0].get("score", 1.0)
    dynamic_threshold = max(min_absolute_score, relative_margin * top_score)
    relevant = [c for c in retrieved_chunks if c.get("score", 1.0) >= dynamic_threshold]
    if not relevant:
        relevant = retrieved_chunks[:1]
    sources = sorted(set(c["title"] for c in relevant))
    citation_block = "\n\nSources:\n" + "\n".join(f"- {s}" for s in sources)
    return answer + citation_block

In [ ]:
!grep -A2 "def format_answer_with_citations" src/generate.py

In [ ]:
import sys
if 'generate' in sys.modules:
    del sys.modules['generate']

from generate import format_answer_with_citations

chunks = [
    {'title': 'What is a Lifetime ISA?', 'score': 0.694},
    {'title': 'What is an ISA?', 'score': 0.547},
    {'title': 'What is a defined benefit pension?', 'score': 0.338},
]
print(format_answer_with_citations("Test", chunks))

In [ ]:
import sys
for mod in ['generate']:
    if mod in sys.modules:
        del sys.modules[mod]

from generate import Generator, format_answer_with_citations
import gradio as gr

generator = Generator()

def answer_question(query, history=None):
    if not query or not query.strip():
        return "Please enter a question."
    retrieved = retrieve(query, index, model, corpus, top_k=3)
    blocked, message = check_safeguards(query, retrieved)
    if blocked:
        return message
    raw_answer = generator.generate(query, retrieved)
    return format_answer_with_citations(raw_answer, retrieved)

demo = gr.ChatInterface(fn=answer_question, title="Finance Guidance Assistant")
demo.launch(share=True)

In [ ]:
!python eval/evaluate.py